# Week 6 · From Answering to Acting — Agent Fundamentals

**GenAI & Agentic AI Programme · 60-minute lab**

---

Over the last two weeks our helpdesk system learned to *answer*:

- **Week 4** — RAG over vectors: "*What is our SLA policy for P1 incidents?*" → grounded answer, citations.
- **Week 5** — RAG over a graph: "*Which services depend on the payment gateway?*" → answers that follow relationships.

Both are read-only. A real helpdesk **acts**: it looks up `INC-1042`, checks whether its SLA
clock has run out, and escalates. No amount of retrieval lets a model *change* anything.
Retrieval fixed what the model **knows**; this week we fix what it can **do**.

**Today: three agents on a difficulty ladder, across two ecosystems.** Single agents only —
teams arrive next week. (Google's agent framework, **ADK**, gets its own dedicated Skill Boost
lab; every concept below transfers to it directly.)

| | 🟢 EASY | 🟡 MODERATE | 🔴 DIFFICULT |
|---|---|---|---|
| **Ecosystem** | OpenAI **Agents SDK** | **Claude** — raw tool-use loop, by hand | OpenAI **Agents SDK**, at scale |
| **Agent** | Service-desk status bot | SLA watchdog | Change-request risk assessor |
| **Tools** | 1 | 3 (one gated write) | 5 (CMDB, calendar, incidents, gated write) |
| **Reasoning** | Single hop | 2–3 chained hops + a judgment call | 4+ hops, evidence synthesis, **multi-turn memory** |
| **The lesson** | *It works* — an agent in ~12 lines | *How it works* — the loop, fully visible | *How it scales* — sessions, auto-schemas, structure |

Plus two **small examples** along the way: typed structured output (🟢 side) and parallel tool
calls in a single model turn (🟡 side).

> ⏱ Pacing: setup 4 · hook 4 · primer 4 · EASY 8 + small ex. 4 · MODERATE 18 + small ex. 4 · DIFFICULT 14 · wrap 4 ≈ 60 min

In [1]:
# ── Setup: pinned installs ────────────────────────────────────────────────────
# Two SDKs only — both authenticate with the API keys you already have.
%pip -q install "openai-agents>=0.17" "anthropic>=0.40"

import logging, warnings
warnings.filterwarnings("ignore")
logging.getLogger("openai.agents").setLevel(logging.ERROR)
print("Installs done.")

Note: you may need to restart the kernel to use updated packages.
Installs done.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ── Display helpers (inlined — this notebook is fully self-contained) ─────────
import json, textwrap, os
from getpass import getpass

W = 100
def _rule(ch="─"): print(ch * W)

def banner(title: str):
    print(); _rule("━"); print(f"  {title}"); _rule("━")

def section(title: str):
    print(); print(f"■ {title}"); _rule()

def observe(text: str):
    print(); print("👀 OBSERVE"); print(textwrap.fill(text, W))

def discuss(text: str):
    print(); print("💬 DISCUSS"); print(textwrap.fill(text, W))

def warn(text: str):
    print(); print("⚠ " + textwrap.fill(text, W - 3))

def success(text: str):
    print(); print("✔ " + textwrap.fill(text, W - 3))

def trace(kind: str, text: str):
    """Plain-text trace of an agent loop — every step of the mechanism, visible."""
    tag = {"model": "  [model]  ", "call": "  → call   ", "result": "  ← result "}[kind]
    body = textwrap.fill(text, W - len(tag), subsequent_indent=" " * len(tag))
    print(tag + body)

def compare(blocks: dict):
    """Print labelled blocks for A/B/C comparison."""
    for label, text in blocks.items():
        print(); print(f"┌─ {label} " + "─" * max(0, W - len(label) - 4))
        print(textwrap.fill(str(text), W - 2, initial_indent="│ ", subsequent_indent="│ "))
    print("└" + "─" * (W - 1))

success("Helpers ready.")


✔ Helpers ready.


In [3]:
# ── Authentication: the two keys you already have ────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

from anthropic import Anthropic
from openai import OpenAI
anthropic_client = Anthropic()
openai_client    = OpenAI()

OPENAI_MODEL = "gpt-4o-mini"
CLAUDE_MODEL = "claude-haiku-4-5"

success("Auth ready.")


✔ Auth ready.


## Step 1 · The hook — ask a plain LLM to do a helpdesk job *(≈4 min)*

Same opening move as Week 3's restaurant question: **look at the output first**, explain the
mechanism after. A strong model, no tools, a perfectly ordinary helpdesk request.

In [4]:
banner("A plain LLM, asked to act")

question = ("What is the current status of incident INC-1042 in our ServiceNow instance, "
            "and is it breaching its SLA? If it is, escalate it to the major incident team.")

resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL, max_tokens=300,
    messages=[{"role": "user", "content": question}],
)
print(resp.choices[0].message.content)

observe("One of two things just happened. Either the model admitted it has no access to your "
        "ServiceNow instance (honest, but useless — the ticket is still burning), or it produced "
        "a confident, plausible-looking status for INC-1042 that it simply invented. Week 3's "
        "restaurant hallucination, wearing a service-desk uniform.")

discuss("RAG cannot rescue us here. Even perfect retrieval only changes what the model KNOWS at "
        "generation time. This request needs the system to LOOK SOMETHING UP in a live record, "
        "EVALUATE it against policy, and CHANGE STATE in another system. Knowing vs doing — "
        "that gap is exactly what an agent closes.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  A plain LLM, asked to act
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
I don't have real-time access to external systems, including ServiceNow, to check the status of specific incidents like INC-1042. To determine the current status of the incident and whether it is breaching its SLA, you would need to log into your ServiceNow instance and perform the following steps:

1. Search for the incident number (INC-1042) in the ServiceNow dashboard.
2. Check the status of the incident and the SLA details.
3. If the SLA is indeed being breached or is at risk of breaching, follow your organization's procedures to escalate the incident to the major incident team.

If you need assistance with how to escalate an incident within ServiceNow, feel free to ask!

👀 OBSERVE
One of two things just happened. Either the model admitted it has no acces

## Concept primer · What is an agent?

Strip away the hype and every agent — OpenAI's, Anthropic's, Google's — is the same three things:

```
            AGENT  =  LLM  +  TOOLS  +  LOOP

   ┌──────────────────────────────────────────────────┐
   │                 while not done:                  │
   │                                                  │
   │   ┌─────────┐   "call get_incident(INC-1042)"    │
   │   │   LLM   │ ───────────────────────────────►   │
   │   │ (plans) │                    ┌────────────┐  │
   │   │         │ ◄───────────────── │ YOUR code  │  │
   │   └─────────┘   {"state": ...}   │ (executes) │  │
   │                                  └────────────┘  │
   └──────────────────────────────────────────────────┘
```

Three framings for an engineering audience:

1. **The LLM never executes anything.** It returns a *function-call intent* — like a service
   responding with an RPC request payload. *Your* runtime validates it, executes it, and posts
   the result back. The model is the planner; your code is the executor.
2. **The loop is literally a `while` loop.** Call model → if it requested a tool, run it, append
   the result, go again → if it produced a final answer, stop. Everything a framework adds
   (sessions, tracing, retries, deployment) wraps around this loop.
3. **Tools are an internal API contract.** A typed function signature plus a one-sentence
   description. The model routes on the description and fills the parameters — a developer
   reading your API docs. The docstring *is* the contract.

Today you'll see this identical loop three ways: hidden by an SDK (EASY), written by hand
(MODERATE), then back to the SDK once the agent is big enough that the machinery earns its keep
(DIFFICULT). Keep asking: **who decides** (the model) vs **who executes** (your code).

---
# 🟢 EASY · OpenAI ecosystem — a status bot in ~12 lines *(≈8 min)*

**Agent:** a service-desk status bot. **Tool:** one — `get_system_status`. **SDK:** the OpenAI
Agents SDK, currently the shortest path from zero to a working agent.

Watch the pattern that repeats all lab: a plain Python function → a schema the model can see →
an agent that decides when to call it. Here the SDK's `function_tool` wrapper generates the
schema from the type hints and docstring, and `Runner` hides the loop entirely.

In [5]:
# ── One tool: a live service-status board ─────────────────────────────────────
SERVICE_STATUS = {
    "payment-gateway": {"service": "payment-gateway", "state": "major_outage",
                        "detail": "Card transactions failing since 09:12 IST (tracked as INC-1042)."},
    "vpn":             {"service": "vpn", "state": "operational",
                        "detail": "All regions normal."},
    "email":           {"service": "email", "state": "degraded",
                        "detail": "EU deliveries delayed ~40 min (tracked as INC-1203)."},
}

def get_system_status(service: str) -> dict:
    """Return the current operational status for a named IT service.

    Args:
        service: Service name, e.g. 'payment-gateway', 'vpn', 'email'.

    Returns:
        dict: status plus the service record, or an error listing known services.
    """
    rec = SERVICE_STATUS.get(service.strip().lower())
    if rec is None:
        return {"status": "error",
                "error_message": f"Unknown service '{service}'. Known: {sorted(SERVICE_STATUS)}"}
    return {"status": "success", "system": rec}

section("A tool is just a function — direct call, no LLM involved")
print(json.dumps(get_system_status("vpn"), indent=2))

# ── The whole agent ───────────────────────────────────────────────────────────
from agents import Agent as OAAgent, Runner as OARunner, function_tool

status_bot = OAAgent(
    name="status_bot",
    model=OPENAI_MODEL,
    instructions=("You are an IT service-desk status bot. Use get_system_status for every "
                  "status question — never guess or answer from memory. If the tool mentions "
                  "a tracking incident ID, include it. Keep answers under 60 words."),
    tools=[function_tool(get_system_status)],   # wrapper builds the schema from hints + docstring
)

success("Agent defined. Count the lines above — that is the entire agent.")


■ A tool is just a function — direct call, no LLM involved
────────────────────────────────────────────────────────────────────────────────────────────────────
{
  "status": "success",
  "system": {
    "service": "vpn",
    "state": "operational",
    "detail": "All regions normal."
  }
}

✔ Agent defined. Count the lines above — that is the entire agent.


In [6]:
def oa_trace(result):
    """Show the loop the SDK ran for us — reused again in the DIFFICULT part."""
    for item in result.new_items:
        t = str(getattr(item, "type", type(item).__name__)).lower()
        if "tool_call_output" in t:
            trace("result", str(item.output))
        elif "tool_call" in t:
            raw = item.raw_item
            trace("call", f"{getattr(raw, 'name', '?')}({getattr(raw, 'arguments', '')})")

# ── Run it. (Notebooks already run an event loop, so: top-level await, ─────────
#    NOT Runner.run_sync — run_sync raises inside Jupyter/Colab.)
result = await OARunner.run(status_bot, "Is the payment gateway up? And what about VPN?")

section("Behind the curtain — the SDK kept a record of the loop it ran for us")
oa_trace(result)

section("Final answer")
print(result.final_output)

observe("Two questions in one prompt → the agent chose to call the tool twice, once per "
        "service, then composed a single answer citing INC-1042. We never told it to call "
        "anything twice — it planned that.")

discuss("Deceptively easy — and that's the point AND the problem. Where is the loop? Where do "
        "tool results become messages? What stops an infinite loop? The SDK answers all of it "
        "for you, which is wonderful right up until you need to debug one. Shortly we build the "
        "identical machinery by hand, with Claude, and watch every step.")


■ Behind the curtain — the SDK kept a record of the loop it ran for us
────────────────────────────────────────────────────────────────────────────────────────────────────
  → call   get_system_status({"service":"payment-gateway"})
  → call   get_system_status({"service":"vpn"})
  ← result {'status': 'success', 'system': {'service': 'payment-gateway', 'state': 'major_outage',
           'detail': 'Card transactions failing since 09:12 IST (tracked as INC-1042).'}}
  ← result {'status': 'success', 'system': {'service': 'vpn', 'state': 'operational', 'detail': 'All
           regions normal.'}}

■ Final answer
────────────────────────────────────────────────────────────────────────────────────────────────────
The payment gateway is experiencing a major outage, affecting card transactions since 09:12 IST (tracked as INC-1042). The VPN service is operational and all regions are normal.

👀 OBSERVE
Two questions in one prompt → the agent chose to call the tool twice, once per service, then


### 🟢 Small example · typed structured output *(≈4 min)*

Free-text answers are for humans. The moment an agent's answer feeds *another system* — a
dashboard, a ticket field, a downstream script — you want a **typed object**, not prose to
regex through.

The Agents SDK takes an `output_type`: give it a dataclass and the final output arrives as an
*instance of your class*, validated. One parameter, and the agent becomes a component you can
program against.

In [7]:
from dataclasses import dataclass

@dataclass
class StatusReport:
    service: str
    state: str                      # operational | degraded | major_outage
    tracking_incident: str | None   # e.g. "INC-1042", or None if healthy
    user_impact_one_liner: str

structured_status_bot = OAAgent(
    name="structured_status_bot",
    model=OPENAI_MODEL,
    instructions=("Report IT service status. Use get_system_status — never guess. "
                  "Fill every field of the report from the tool result."),
    tools=[function_tool(get_system_status)],
    output_type=StatusReport,                    # ← the one-line upgrade
)

result = await OARunner.run(structured_status_bot, "Status report for the payment gateway, please.")
report = result.final_output                     # a real StatusReport instance

section("Not prose — a typed object")
print(type(report).__name__, "→", report)
print("\nField access, like any Python object:  report.state =", repr(report.state))

observe("Same tool, same facts — but the answer is now report.state and "
        "report.tracking_incident, ready for an if-statement or a dashboard, with zero string "
        "parsing. Compare with the MODERATE part coming up, where we enforce structure the "
        "manual way: prompt rules plus validation code.")


■ Not prose — a typed object
────────────────────────────────────────────────────────────────────────────────────────────────────
StatusReport → StatusReport(service='payment-gateway', state='major_outage', tracking_incident='INC-1042', user_impact_one_liner='Card transactions failing since 09:12 IST.')

Field access, like any Python object:  report.state = 'major_outage'

👀 OBSERVE
Same tool, same facts — but the answer is now report.state and report.tracking_incident, ready for
an if-statement or a dashboard, with zero string parsing. Compare with the MODERATE part coming up,
where we enforce structure the manual way: prompt rules plus validation code.


---
# 🟡 MODERATE · Claude — the raw agent loop, by hand *(≈18 min)*

**Agent:** an SLA watchdog. **Tools:** three — a read, a policy lookup, and a **gated write**.
**SDK:** none. Just the Anthropic Messages API and a `while` loop we write ourselves.

This is the teaching centrepiece: the mechanism the EASY part hid, fully visible. Tool-writing
rules we follow (they pay off again in the DIFFICULT part):

- **Full type hints** on every parameter — schemas are generated from them (here, by us).
- **One clear docstring sentence first** — the LLM routes on it.
- **Return a typed `dict` with a `status` key**, never a raw string — models parse structure
  reliably, prose unreliably. Tools return structured *errors* rather than raising, so the
  agent can read the error and recover instead of crashing the loop.
- `escalate_incident` **mutates state**. In production a destructive tool sits behind a
  change-request gate and human approval — we flag the spot today; **Week 7 builds the gate.**

In [8]:
from datetime import datetime, timedelta

# ── Mock ServiceNow tables ────────────────────────────────────────────────────
def _fresh_incidents() -> dict:
    """(Re)build the incident table. opened_at is computed relative to *now*, so
    INC-1042 is ALWAYS 5h old and breaching, whatever day you run this lab."""
    now = datetime.now()
    return {
        "INC-1042": {"incident_id": "INC-1042", "priority": "P1", "state": "In Progress",
                     "short_description": "Payment gateway down — all card transactions failing",
                     "assigned_group": "Payments L2", "opened_at": now - timedelta(hours=5)},
        "INC-1105": {"incident_id": "INC-1105", "priority": "P4", "state": "New",
                     "short_description": "Request: replacement mouse for workstation BLR-2214",
                     "assigned_group": "Desktop Support", "opened_at": now - timedelta(hours=2)},
        "INC-1203": {"incident_id": "INC-1203", "priority": "P2", "state": "In Progress",
                     "short_description": "Email delivery delayed for EU region users",
                     "assigned_group": "Messaging Ops", "opened_at": now - timedelta(hours=3)},
    }

SLA_POLICIES = {
    "P1": {"priority": "P1", "resolution_target_hours": 4,
           "description": "Critical — business-stopping outage. Resolve within 4 hours, 24x7."},
    "P2": {"priority": "P2", "resolution_target_hours": 8,
           "description": "High — major degradation. Resolve within 8 business hours."},
    "P3": {"priority": "P3", "resolution_target_hours": 24,
           "description": "Moderate — limited impact. Resolve within 24 business hours."},
    "P4": {"priority": "P4", "resolution_target_hours": 48,
           "description": "Low — requests and minor issues. Resolve within 48 business hours."},
}

INCIDENTS  = _fresh_incidents()
AUDIT_LOG: list[dict] = []          # shared by every write-tool in this lab

def reset_incident_backend():
    """Idempotent labs: restore pristine state before each agent run."""
    global INCIDENTS
    INCIDENTS = _fresh_incidents()
    AUDIT_LOG.clear()

# ── The three tools ───────────────────────────────────────────────────────────
def get_incident(incident_id: str) -> dict:
    """Look up a ServiceNow incident record by its incident number (e.g. 'INC-1042').

    Args:
        incident_id: The incident number to look up.

    Returns:
        dict: status plus the incident record (including age_hours), or an error message.
    """
    record = INCIDENTS.get(incident_id.strip().upper())
    if record is None:
        return {"status": "error", "error_message": f"No incident found with ID '{incident_id}'."}
    age_hours = round((datetime.now() - record["opened_at"]).total_seconds() / 3600, 1)
    rec = {**record, "opened_at": record["opened_at"].isoformat(timespec="minutes"),
           "age_hours": age_hours}
    return {"status": "success", "incident": rec}

def check_sla_policy(priority: str) -> dict:
    """Return the SLA resolution policy for a given incident priority (P1–P4).

    Args:
        priority: Incident priority level: 'P1', 'P2', 'P3' or 'P4'.

    Returns:
        dict: status plus the policy (resolution_target_hours, description), or an error message.
    """
    policy = SLA_POLICIES.get(priority.strip().upper())
    if policy is None:
        return {"status": "error", "error_message": f"Unknown priority '{priority}'. Use P1–P4."}
    return {"status": "success", "policy": policy}

def escalate_incident(incident_id: str, reason: str) -> dict:
    """Escalate an incident to the Major Incident Team. Use ONLY when an SLA breach is confirmed.

    Args:
        incident_id: The incident number to escalate.
        reason: One-sentence justification citing the SLA breach.

    Returns:
        dict: status plus the updated record, or an error message.
    """
    # ⚠ DESTRUCTIVE TOOL — mutates state. In production this is where a change-request
    # gate + human approval belongs (Week 7). Today: guard rails + audit log.
    record = INCIDENTS.get(incident_id.strip().upper())
    if record is None:
        return {"status": "error", "error_message": f"No incident found with ID '{incident_id}'."}
    if record["state"] == "Escalated":
        return {"status": "error", "error_message": f"{incident_id} is already escalated."}
    if not reason or len(reason.strip()) < 10:
        return {"status": "error", "error_message": "A meaningful escalation reason is required."}
    record["state"] = "Escalated"
    record["assigned_group"] = "Major Incident Team"
    AUDIT_LOG.append({"ts": datetime.now().isoformat(timespec="seconds"), "actor": "agent",
                      "action": "escalate_incident", "target": record["incident_id"],
                      "reason": reason.strip()})
    return {"status": "success",
            "incident": {k: v for k, v in record.items() if k != "opened_at"},
            "message": f"{record['incident_id']} escalated to Major Incident Team."}

section("Direct calls — no LLM involved yet")
print(json.dumps(get_incident("INC-1042"), indent=2))
print(json.dumps(check_sla_policy("P1"), indent=2))
print(json.dumps(get_incident("INC-9999"), indent=2))   # the error path — deliberately

discuss("INC-1042 is a P1 open ~5 hours against a 4-hour SLA — a breach, by design. And note "
        "the error path: a structured error, not an exception. The agent gets to READ the "
        "failure and recover.")


■ Direct calls — no LLM involved yet
────────────────────────────────────────────────────────────────────────────────────────────────────
{
  "status": "success",
  "incident": {
    "incident_id": "INC-1042",
    "priority": "P1",
    "state": "In Progress",
    "short_description": "Payment gateway down \u2014 all card transactions failing",
    "assigned_group": "Payments L2",
    "opened_at": "2026-07-05T07:56",
    "age_hours": 5.0
  }
}
{
  "status": "success",
  "policy": {
    "priority": "P1",
    "resolution_target_hours": 4,
    "description": "Critical \u2014 business-stopping outage. Resolve within 4 hours, 24x7."
  }
}
{
  "status": "error",
  "error_message": "No incident found with ID 'INC-9999'."
}

💬 DISCUSS
INC-1042 is a P1 open ~5 hours against a 4-hour SLA — a breach, by design. And note the error path:
a structured error, not an exception. The agent gets to READ the failure and recover.


### Tool schemas · how the model sees your functions

The model never sees your Python. It sees a **JSON Schema**: name, description, typed
parameters. When it wants a tool it replies with a `tool_use` block — name + arguments — and
*stops*. Your loop does the rest.

Schema authoring is the tell that distinguishes the three levels today:

| 🟢 EASY | 🟡 MODERATE | 🔴 DIFFICULT |
|---|---|---|
| `function_tool` generated it | **we write it by hand, once, so you see the wire format** | `function_tool` again — for five tools, generation earns its keep |

In [12]:
CLAUDE_TOOLS = [
    {
        "name": "get_incident",
        "description": "Look up a ServiceNow incident record by its incident number (e.g. 'INC-1042').",
        "input_schema": {
            "type": "object",
            "properties": {
                "incident_id": {"type": "string", "description": "The incident number to look up."}
            },
            "required": ["incident_id"],
        },
    },
    {
        "name": "check_sla_policy",
        "description": "Return the SLA resolution policy for a given incident priority (P1-P4).",
        "input_schema": {
            "type": "object",
            "properties": {
                "priority": {"type": "string", "description": "Incident priority: P1, P2, P3 or P4."}
            },
            "required": ["priority"],
        },
    },
    {
        "name": "escalate_incident",
        "description": "Escalate an incident to the Major Incident Team. Use ONLY when an SLA breach is confirmed.",
        "input_schema": {
            "type": "object",
            "properties": {
                "incident_id": {"type": "string", "description": "The incident number to escalate."},
                "reason": {"type": "string", "description": "One-sentence justification citing the SLA breach."},
            },
            "required": ["incident_id", "reason"],
        },
    },
]

# Name -> Python function. The model outputs a NAME; only this registry maps names to code.
TOOL_REGISTRY = {
    "get_incident": get_incident,
    "check_sla_policy": check_sla_policy,
    "escalate_incident": escalate_incident,
}

section("What actually crosses the wire for one tool")
print(json.dumps(CLAUDE_TOOLS[1], indent=2))

discuss("The registry is a security boundary, not a convenience. The model can only NAME tools; "
        "it cannot inject code. If a name isn't in the registry, nothing runs. Keep that mental "
        "model — it carries straight into MCP in Week 8.")


■ What actually crosses the wire for one tool
────────────────────────────────────────────────────────────────────────────────────────────────────
{
  "name": "check_sla_policy",
  "description": "Return the SLA resolution policy for a given incident priority (P1-P4).",
  "input_schema": {
    "type": "object",
    "properties": {
      "priority": {
        "type": "string",
        "description": "Incident priority: P1, P2, P3 or P4."
      }
    },
    "required": [
      "priority"
    ]
  }
}

💬 DISCUSS
The registry is a security boundary, not a convenience. The model can only NAME tools; it cannot
inject code. If a name isn't in the registry, nothing runs. Keep that mental model — it carries
straight into MCP in Week 8.


In [13]:
SLA_AGENT_PROMPT = """You are an IT helpdesk SLA watchdog for our ServiceNow instance.

Rules:
1. Never invent incident data. Every fact about an incident MUST come from a tool result.
2. To judge an SLA breach, compare the incident's age_hours against the policy's
   resolution_target_hours for its priority. State both numbers in your answer.
3. Escalate ONLY when a breach is confirmed by the numbers. If there is no breach, say so
   and do not escalate.
4. If a tool returns status='error', explain the problem to the user; do not retry blindly.
5. Cite incident IDs (e.g. INC-1042) in your final answer, and keep it under 120 words."""

def run_claude_agent(user_query: str, max_iterations: int = 8) -> str:
    """A complete agent, by hand: Claude + tools + a while loop."""
    reset_incident_backend()              # idempotent runs — every demo starts pristine
    messages = [{"role": "user", "content": user_query}]
    banner(f"RAW AGENT LOOP · {user_query}")

    for iteration in range(1, max_iterations + 1):
        response = anthropic_client.messages.create(
            model=CLAUDE_MODEL, max_tokens=1024,
            system=SLA_AGENT_PROMPT, tools=CLAUDE_TOOLS, messages=messages,
        )

        for block in response.content:                       # model may think out loud
            if block.type == "text" and block.text.strip():
                trace("model", block.text.strip())

        if response.stop_reason != "tool_use":                # end_turn -> we're done
            final = "".join(b.text for b in response.content if b.type == "text")
            success(f"Final answer after {iteration} model call(s).")
            return final

        # The model requested tool(s). Keep its turn, execute, post results back.
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                trace("call", f"{block.name}({json.dumps(block.input)})")
                fn = TOOL_REGISTRY.get(block.name)            # registry = the security boundary
                result = fn(**block.input) if fn else {"status": "error",
                                                       "error_message": f"Unknown tool {block.name}"}
                trace("result", json.dumps(result, default=str))
                tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                     "content": json.dumps(result, default=str)})
        messages.append({"role": "user", "content": tool_results})

    warn("Max iterations reached — loop terminated by the guard rail, not by the model.")
    return "(no final answer)"

print("run_claude_agent() defined — the entire agent is the ~40 lines above.")
print("Guard rails to point at: max_iterations (YOUR runtime owns termination),")
print("stop_reason (the only branch), tool_result tied to the request id (correlation).")

run_claude_agent() defined — the entire agent is the ~40 lines above.
Guard rails to point at: max_iterations (YOUR runtime owns termination),
stop_reason (the only branch), tool_result tied to the request id (correlation).


In [14]:
# ── Run 1: the question the plain LLM couldn't touch ─────────────────────────
answer = run_claude_agent(
    "Is INC-1042 breaching its SLA? If it is, escalate it with a proper reason."
)
section("Agent's final answer")
print(answer)

observe("Read the trace top to bottom: the model FETCHED the incident, FETCHED the P1 policy, "
        "COMPARED 5h against 4h, then — and only then — called escalate_incident with a reason "
        "citing the breach. A plan it composed itself; we never told it which tools to use or "
        "in what order.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RAW AGENT LOOP · Is INC-1042 breaching its SLA? If it is, escalate it with a proper reason.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [model]  I'll check the incident details and compare them against the SLA policy.
  → call   get_incident({"incident_id": "INC-1042"})
  ← result {"status": "success", "incident": {"incident_id": "INC-1042", "priority": "P1", "state":
           "In Progress", "short_description": "Payment gateway down \u2014 all card
           transactions failing", "assigned_group": "Payments L2", "opened_at":
           "2026-07-05T08:21", "age_hours": 5.0}}
  [model]  Now let me check the SLA policy for P1 priority:
  → call   check_sla_policy({"priority": "P1"})
  ← result {"status": "success", "policy": {"priority": "P1", "resolution_target_hours": 4,
           "description": "Critical \u2014 business

In [15]:
# ── Run 2: judgment, not reflex — a P4 that is NOT breaching ──────────────────
answer = run_claude_agent("Is INC-1105 breaching its SLA? Escalate it if needed.")
section("Agent's final answer")
print(answer)

# ── Run 3: the error path — an incident that doesn't exist ────────────────────
answer = run_claude_agent("Check INC-9999 and escalate it if it's breaching.")
section("Agent's final answer")
print(answer)

observe("Run 2: 2h old against a 48h target — the agent checked and correctly REFUSED to "
        "escalate. Run 3: the tool returned status='error' and rule 4 turned that into a "
        "graceful explanation instead of a hallucinated ticket or a crashed loop. Deciding "
        "NOT to act is agent behaviour too.")

discuss("Notice where the intelligence lives and where the safety lives. Intelligence: the "
        "model chose the tool sequence and made the breach judgment. Safety: deterministic "
        "code — the registry, the max-iteration guard, the reason-length check inside "
        "escalate_incident. Never delegate the safety half to the model.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RAW AGENT LOOP · Is INC-1105 breaching its SLA? Escalate it if needed.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [model]  I'll check the incident details and compare them against the SLA policy.
  → call   get_incident({"incident_id": "INC-1105"})
  ← result {"status": "success", "incident": {"incident_id": "INC-1105", "priority": "P4", "state":
           "New", "short_description": "Request: replacement mouse for workstation
           BLR-2214", "assigned_group": "Desktop Support", "opened_at":
           "2026-07-05T11:22", "age_hours": 2.0}}
  [model]  Now let me check the SLA policy for P4 priority:
  → call   check_sla_policy({"priority": "P4"})
  ← result {"status": "success", "policy": {"priority": "P4", "resolution_target_hours": 48,
           "description": "Low \u2014 requests and minor issues. Resolve within 4

### 🟡 Small example · parallel tool calls in ONE model turn *(≈4 min)*

Look closely at our loop: it iterates over **every** `tool_use` block in a response before
replying. That's not defensive coding — the model can request *several tools in a single turn*
when the calls don't depend on each other.

Ask about two unrelated incidents at once and watch one model call fan out into two tool
executions. In production, that's your cue to run independent calls concurrently — one round
trip to the model instead of two.

In [16]:
answer = run_claude_agent(
    "Give me a one-line status for INC-1042 and a one-line status for INC-1203. "
    "Just status — no escalation analysis."
)
section("Agent's final answer")
print(answer)

observe("Check the trace: if both get_incident calls appear BEFORE the first '← result', they "
        "arrived in one model turn — one plan, two calls, batched. (Models decide this "
        "themselves; if it chose sequential calls this run, rerun the cell and compare.) Our "
        "loop needed zero changes to support it — because it always treated 'the model's turn' "
        "as a LIST of requests, not a single one.")

discuss("This is why the tool_result blocks carry the request's tool_use_id: with several "
        "calls in flight, results must be correlated back to the right request — the same "
        "reason async APIs carry request IDs. Frameworks inherit this behaviour silently; now "
        "you know what they're inheriting.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RAW AGENT LOOP · Give me a one-line status for INC-1042 and a one-line status for INC-1203. Just status — no escalation analysis.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  → call   get_incident({"incident_id": "INC-1042"})
  ← result {"status": "success", "incident": {"incident_id": "INC-1042", "priority": "P1", "state":
           "In Progress", "short_description": "Payment gateway down \u2014 all card
           transactions failing", "assigned_group": "Payments L2", "opened_at":
           "2026-07-05T08:23", "age_hours": 5.0}}
  → call   get_incident({"incident_id": "INC-1203"})
  ← result {"status": "success", "incident": {"incident_id": "INC-1203", "priority": "P2", "state":
           "In Progress", "short_description": "Email delivery delayed for EU region
           users", "assigned_group": "Messaging Ops", "open

### Anatomy recap — you have now built an agent

```
messages = [user query]
while iterations remain:
    response = model(system, tools, messages)     # model PLANS
    if response.stop_reason != "tool_use": done   # model finished
    for each tool_use block:                      # may be SEVERAL (parallel calls)
        result = REGISTRY[name](**args)           # YOUR code EXECUTES
    messages += [assistant turn, tool results]    # post results, loop
```

That's the entire pattern — and it is *exactly* what the EASY part's `Runner` did behind the
curtain. What frameworks sell is everything around this loop: schema generation, session state,
observability, deployment. Time to make the frameworks earn it: same SDK as the EASY part, but
an agent big enough to need it.

---
# 🔴 DIFFICULT · OpenAI Agents SDK, at scale — a change-risk assessor *(≈14 min)*

**Agent:** a change-request risk assessor. **Tools:** five. **New machinery:** a **session** —
persistent conversation memory across turns, something neither agent so far possessed.

**"Difficult" here means the *agent* is difficult, not the code.** The scenario:

> `CHG-2041` proposes patching the **payment-gateway** cluster **Friday 18:00**. Should it
> proceed?

A competent assessor must: pull the CR → walk the **service dependency graph** to establish
blast radius (hello, Week 5 — that's a graph traversal in agent clothing) → check the change
**freeze calendar** → check **recent incidents** on the target *and* every dependent service →
synthesize a risk verdict → record it with cited evidence. That's 4+ tool hops with a judgment
at the end — and then we ask a **follow-up in the same session** and watch the agent reuse what
it already learned.

Note what the five tools below are: *plain functions with the exact same docstring discipline
as the MODERATE part* — no hand-written schemas this time. Five tools is where generation stops
being a convenience and starts being the only sane option. (In the dedicated Skill Boost lab
you'll see Google's ADK make the identical move: bare functions in, schemas out.)

In [17]:
# ── Change-management backend: CMDB slice, freeze calendar, incident history ──
def _fresh_changes() -> dict:
    return {
        "CHG-2041": {"change_id": "CHG-2041",
                     "summary": "Apply security patch KB-88231 to payment-gateway cluster",
                     "target_service": "payment-gateway",
                     "proposed_day": "Friday", "proposed_start_hour": 18,
                     "duration_hours": 2, "requester": "Payments L2",
                     "backout_plan": True, "risk_flag": None},
    }

CHANGE_REQUESTS = _fresh_changes()

def reset_change_backend():
    global CHANGE_REQUESTS
    CHANGE_REQUESTS = _fresh_changes()
    AUDIT_LOG.clear()

SERVICE_DEPENDENCIES = {          # who sits DOWNSTREAM (breaks if this service breaks)
    "payment-gateway": ["checkout-web", "mobile-app", "partner-api"],
    "checkout-web": [], "mobile-app": [], "partner-api": [], "email": [], "vpn": [],
}

CHANGE_WINDOWS = {                # approved standard-change windows, 24h clock
    "monday": (22, 6), "tuesday": (22, 6), "wednesday": (22, 6), "thursday": (22, 6),
    "friday": None,               # freeze — peak transaction window
    "saturday": (0, 24), "sunday": (0, 24),
}

RECENT_INCIDENTS = {              # last 30 days, per service
    "payment-gateway": [
        {"incident_id": "INC-1042", "priority": "P1", "state": "Open",
         "summary": "Card transactions failing"},
        {"incident_id": "INC-0991", "priority": "P2", "state": "Resolved",
         "summary": "Intermittent 5xx from gateway API"}],
    "checkout-web": [{"incident_id": "INC-0975", "priority": "P3", "state": "Resolved",
                      "summary": "Slow page loads"}],
    "mobile-app": [],
    "partner-api": [{"incident_id": "INC-1010", "priority": "P2", "state": "Resolved",
                     "summary": "Webhook retries exhausted"}],
}

# ── Five tools. Same docstring discipline — the SDK generates the schemas. ────
def get_change_request(change_id: str) -> dict:
    """Look up a change request record by its ID (e.g. 'CHG-2041').

    Args:
        change_id: The change request number to look up.

    Returns:
        dict: status plus the change record, or an error message.
    """
    rec = CHANGE_REQUESTS.get(change_id.strip().upper())
    if rec is None:
        return {"status": "error", "error_message": f"No change request '{change_id}'."}
    return {"status": "success", "change": rec}

def get_dependent_services(service: str) -> dict:
    """List the services directly downstream of a service (its blast radius if it fails).

    Args:
        service: The service name, e.g. 'payment-gateway'.

    Returns:
        dict: status plus the downstream service list and blast radius count.
    """
    if service.strip().lower() not in SERVICE_DEPENDENCIES:
        return {"status": "error", "error_message": f"Service '{service}' not in the CMDB."}
    downstream = SERVICE_DEPENDENCIES[service.strip().lower()]
    return {"status": "success", "service": service, "downstream": downstream,
            "blast_radius": len(downstream)}

def check_change_window(day: str, start_hour: int) -> dict:
    """Check whether a proposed weekday and start hour falls inside an approved change window.

    Args:
        day: Weekday name, e.g. 'Friday'.
        start_hour: Proposed start hour, 24h clock (0-23).

    Returns:
        dict: status plus the window verdict (open true/false with reason or approved hours).
    """
    key = day.strip().lower()
    if key not in CHANGE_WINDOWS:
        return {"status": "error", "error_message": f"Unknown weekday '{day}'."}
    win = CHANGE_WINDOWS[key]
    if win is None:
        return {"status": "success", "window": {"day": day, "open": False,
                "reason": "Change freeze: Friday is the peak transaction window; "
                          "standard changes are not permitted."}}
    start, end = win
    inside = (start <= start_hour or start_hour < end) if start > end \
             else (start <= start_hour < end)
    verdict = {"day": day, "open": inside,
               "approved_hours": f"{start:02d}:00–{(end % 24):02d}:00"}
    if not inside:
        verdict["reason"] = f"{start_hour}:00 is outside the approved window."
    return {"status": "success", "window": verdict}

def get_recent_incidents(service: str) -> dict:
    """Return the last 30 days of incidents for a service, with an open-P1 count.

    Args:
        service: The service name, e.g. 'payment-gateway'.

    Returns:
        dict: status plus incident list and open_p1_count for quick risk checks.
    """
    if service.strip().lower() not in SERVICE_DEPENDENCIES:
        return {"status": "error", "error_message": f"Service '{service}' not in the CMDB."}
    incidents = RECENT_INCIDENTS.get(service.strip().lower(), [])
    open_p1 = sum(1 for i in incidents if i["priority"] == "P1" and i["state"] == "Open")
    return {"status": "success", "service": service,
            "open_p1_count": open_p1, "incidents": incidents}

def flag_change(change_id: str, risk_level: str, justification: str) -> dict:
    """Record a risk verdict on a change request. Requires evidence-based justification.

    Args:
        change_id: The change request number to flag.
        risk_level: One of 'low', 'medium', 'high'.
        justification: Evidence citing window status, incident IDs and blast radius (min 20 chars).

    Returns:
        dict: status plus the updated change record, or an error message.
    """
    # ⚠ WRITE TOOL — same discipline as escalate_incident: validated, audited, and in
    # production, gated behind human approval (Week 7).
    rec = CHANGE_REQUESTS.get(change_id.strip().upper())
    if rec is None:
        return {"status": "error", "error_message": f"No change request '{change_id}'."}
    if risk_level.strip().lower() not in ("low", "medium", "high"):
        return {"status": "error", "error_message": "risk_level must be low, medium or high."}
    if not justification or len(justification.strip()) < 20:
        return {"status": "error", "error_message": "An evidence-based justification is required."}
    rec["risk_flag"] = {"risk_level": risk_level.strip().lower(),
                        "justification": justification.strip()}
    AUDIT_LOG.append({"ts": datetime.now().isoformat(timespec="seconds"), "actor": "agent",
                      "action": "flag_change", "target": rec["change_id"],
                      "reason": f"{risk_level.lower()}: {justification.strip()}"})
    return {"status": "success", "change": rec,
            "message": f"{rec['change_id']} flagged {risk_level.lower()}."}

section("Direct calls — the CMDB slice and the freeze calendar")
print(json.dumps(get_dependent_services("payment-gateway"), indent=2))
print(json.dumps(check_change_window("Friday", 18), indent=2))

discuss("get_dependent_services is Week 5's graph traversal wearing a tool costume — one hop "
        "along CMDB edges. Next week's ServiceNow-integrated agents will walk the real thing. "
        "And Friday 18:00 is frozen, by design: the agent will have to discover that itself.")


■ Direct calls — the CMDB slice and the freeze calendar
────────────────────────────────────────────────────────────────────────────────────────────────────
{
  "status": "success",
  "service": "payment-gateway",
  "downstream": [
    "checkout-web",
    "mobile-app",
    "partner-api"
  ],
  "blast_radius": 3
}
{
  "status": "success",
  "window": {
    "day": "Friday",
    "open": false,
    "reason": "Change freeze: Friday is the peak transaction window; standard changes are not permitted."
  }
}

💬 DISCUSS
get_dependent_services is Week 5's graph traversal wearing a tool costume — one hop along CMDB
edges. Next week's ServiceNow-integrated agents will walk the real thing. And Friday 18:00 is
frozen, by design: the agent will have to discover that itself.


In [18]:
from agents import SQLiteSession

RISK_AGENT_INSTRUCTION = """You are a change-management risk assessor for our IT estate.

Assessment procedure — follow it in order, using tools for every fact:
1. get_change_request for the CR under review.
2. get_dependent_services for the target service, to establish the blast radius.
3. check_change_window for the proposed day and start hour.
4. get_recent_incidents for the target service AND for each dependent service.
5. Decide the risk level:
   - high:   the window is closed, OR any open P1 exists on the target service.
   - medium: no high trigger, but resolved P1/P2 incidents in the last 30 days on the
             target or any dependent service.
   - low:    none of the above.
6. Record the verdict with flag_change, citing the evidence (window status, incident IDs,
   blast radius) in the justification.

Rules: never invent data — every fact must come from a tool result. Cite CHG and INC numbers.
For a follow-up about the same change in this session, reuse what you already learned and
only re-check what changed. Final answers under 150 words."""

risk_agent = OAAgent(
    name="change_risk_assessor",
    model=OPENAI_MODEL,
    instructions=RISK_AGENT_INSTRUCTION,
    tools=[function_tool(f) for f in (get_change_request, get_dependent_services,
                                      check_change_window, get_recent_incidents, flag_change)],
)

# A session = the conversation's memory, kept OUTSIDE any single run. In-memory SQLite here;
# point db_path at a file (or a server) and the same conversation survives restarts.
chg_session = SQLiteSession("chg-2041-review")

success("Risk agent ready: 5 bare functions in, 5 schemas out — plus a session for memory. "
        "Note what we did NOT write this time: schemas, a loop, message bookkeeping.")


✔ Risk agent ready: 5 bare functions in, 5 schemas out — plus a session for memory. Note what we
did NOT write this time: schemas, a loop, message bookkeeping.


In [19]:
# ── Turn 1: the full assessment ───────────────────────────────────────────────
reset_change_backend()

result = await OARunner.run(risk_agent,
                            "Assess CHG-2041 and flag it with a risk level.",
                            session=chg_session)
banner("TURN 1 · full assessment")
oa_trace(result)                       # same helper from the EASY part
section("Agent's final answer")
print(result.final_output)

observe("Count the hops in the trace: CR → blast radius (3 downstream services) → Friday 18:00 "
        "window (frozen) → incident history for payment-gateway AND its dependents (finding the "
        "open P1, INC-1042) → flag high, with the evidence in the justification. That plan came "
        "from the procedure in the instructions — the agent operationalised it, tool by tool.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TURN 1 · full assessment
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  → call   get_change_request({"change_id":"CHG-2041"})
  ← result {'status': 'success', 'change': {'change_id': 'CHG-2041', 'summary': 'Apply security
           patch KB-88231 to payment-gateway cluster', 'target_service': 'payment-
           gateway', 'proposed_day': 'Friday', 'proposed_start_hour': 18,
           'duration_hours': 2, 'requester': 'Payments L2', 'backout_plan': True,
           'risk_flag': {'risk_level': 'high', 'justification': 'Change window closed
           (Friday peak hours); open P1 incident (INC-1042) on payment-gateway; blast
           radius includes 3 downstream services: checkout-web, mobile-app, partner-
           api.'}}}
  → call   get_dependent_services({"service":"payment-gateway"})
  ← result {'status': 'success', 'ser

In [20]:
# ── Turn 2: SAME session — the capability the other two agents lacked ─────────
result = await OARunner.run(risk_agent,
                            "What if we moved it to Saturday at 10:00 instead? "
                            "Reassess and update the flag.",
                            session=chg_session)
banner("TURN 2 · follow-up in the same session")
oa_trace(result)
section("Agent's final answer")
print(result.final_output)

observe("The agent did not re-ask what 'it' means and did not re-fetch everything — the session "
        "replayed turn 1's history, so it remembered CHG-2041 and its evidence, re-checked only "
        "the window for Saturday 10:00 (open), weighed the still-open P1 on the target, and "
        "revised the verdict. THAT is session state doing real work.")

discuss("Try this follow-up on the MODERATE loop: it would stare blankly at 'it' — our messages "
        "list dies when run_claude_agent returns. Nothing stops you persisting that list "
        "yourself (that's literally all a session is), but bookkeeping it correctly across "
        "turns, users and restarts is exactly the kind of machinery you buy a framework for.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TURN 2 · follow-up in the same session
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  → call   check_change_window({"day":"Saturday","start_hour":10})
  → call   get_recent_incidents({"service":"payment-gateway"})
  → call   get_recent_incidents({"service":"checkout-web"})
  → call   get_recent_incidents({"service":"mobile-app"})
  → call   get_recent_incidents({"service":"partner-api"})
  ← result {'status': 'success', 'window': {'day': 'Saturday', 'open': True, 'approved_hours':
           '00:00–00:00'}}
  ← result {'status': 'success', 'service': 'payment-gateway', 'open_p1_count': 1, 'incidents':
           [{'incident_id': 'INC-1042', 'priority': 'P1', 'state': 'Open', 'summary':
           'Card transactions failing'}, {'incident_id': 'INC-0991', 'priority': 'P2',
           'state': 'Resolved', 'summary': 'Intermittent 5

### Discuss · what the framework bought, at scale

**Bought:** five schemas generated from the docstrings you wrote (the MODERATE discipline paid
off instantly), a managed loop with the run history exposed through `result.new_items`, and
genuine **multi-turn session memory** — swap the in-memory SQLite for a file or server and the
conversation survives restarts, without touching agent code.

**Paid:** ceremony and opacity. An `Agent`, a `Runner`, a `Session`, and a loop you can inspect
but not step through — versus the MODERATE part's forty flat lines you own completely. That is
the standing trade-off: **control vs ceremony**. The raw loop is nicer for one agent and three
tools; the framework wins the moment you need persistence, five-plus tools, or teams of agents.

**Unchanged:** the mechanism. Nothing here added intelligence. If you can read `oa_trace`
output — and after the MODERATE part, you can — you can debug any agent framework, because you
know what the events *are*. The same holds in the ADK Skill Boost lab: different names
(`Runner`, `SessionService`, event stream), identical loop.

In [21]:
section("The audit trail — every state change the risk agent made")
if AUDIT_LOG:
    for entry in AUDIT_LOG:
        print(json.dumps(entry, indent=2))
else:
    print("(empty — rerun the turn-cells above; reset_change_backend clears the log)")

discuss("Two entries if both turns ran: the high flag from turn 1 and the revised verdict from "
        "turn 2 — a decision HISTORY, not just a final state. Three fields do most of the "
        "compliance work: WHO acted (actor), WHAT changed (action + target), and WHY (reason — "
        "which the model itself was forced to supply, and which our code refused to accept "
        "when too thin). Week 7 turns this list into a real, queryable log with human approval "
        "gates in front of it.")


■ The audit trail — every state change the risk agent made
────────────────────────────────────────────────────────────────────────────────────────────────────
{
  "ts": "2026-07-05T13:29:50",
  "actor": "agent",
  "action": "flag_change",
  "target": "CHG-2041",
  "reason": "high: Change window closed (Friday peak hours); open P1 incident (INC-1042) on payment-gateway; blast radius includes 3 downstream services: checkout-web, mobile-app, partner-api."
}
{
  "ts": "2026-07-05T13:30:51",
  "actor": "agent",
  "action": "flag_change",
  "target": "CHG-2041",
  "reason": "medium: Change window open on Saturday; however, open P1 incident (INC-1042) exists on payment-gateway. Blast radius includes 3 downstream services: checkout-web, mobile-app, partner-api."
}

💬 DISCUSS
Two entries if both turns ran: the high flag from turn 1 and the revised verdict from turn 2 — a
decision HISTORY, not just a final state. Three fields do most of the compliance work: WHO acted
(actor), WHAT changed (act

In [ ]:
compare({
    "🟢 EASY · OpenAI Agents SDK, one tool":
        "~12 lines. Schema from function_tool; loop fully hidden; one-shot (no memory). "
        "Fastest zero-to-agent path — and the small structured-output example showed the "
        "answer can be a typed object, not prose.",
    "🟡 MODERATE · Claude, raw Messages-API loop":
        "~40 lines you own end-to-end. Hand-written schemas, hand-written while-loop, print "
        "tracing, explicit guard rails (registry, max-iterations, gated write) — and it "
        "handled parallel tool calls for free because it treats the model's turn as a list. "
        "The mechanism every framework runs underneath.",
    "🔴 DIFFICULT · OpenAI Agents SDK, five tools + session":
        "The same SDK as EASY, now earning its keep: five docstring-generated schemas, a "
        "managed loop with inspectable run items, and session memory across turns. The point "
        "where framework ceremony becomes cheaper than doing it yourself.",
})

discuss("One mechanism, three packagings. If Week 4's lesson was 'grounding beats knowing', "
        "this week's is 'the loop is simple — the engineering around the loop is the product'. "
        "Pick the packaging by what you need AROUND the loop. (You'll apply the same test to "
        "Google's ADK in its Skill Boost lab.)")

## Production notes · the parts we deliberately deferred

Each of these is *absent* today on purpose — and each is a Week 7 or Week 8 topic:

1. **Destructive tools need gates.** `escalate_incident` and `flag_change` mutated state on the
   model's say-so. In production, write tools demand a valid change-request number and, for
   high-impact actions, a human approval step *before* execution. → **Week 7: human-in-the-loop.**
2. **Audit everything.** Our toy `AUDIT_LOG` becomes a real, queryable trail — because "the AI
   did it" is not an acceptable RCA line. → **Week 7: audit logging.**
3. **One agent, one responsibility.** The risk assessor both investigates and records verdicts.
   At scale you split: a triage agent, an analyst, an executor holding the dangerous tools —
   and a coordinator. → **Week 7: multi-agent systems** (with Google's ADK — whose graph-based
   Workflow runtime exists for exactly this — covered hands-on in its Skill Boost lab).
4. **Hand-registered tools don't scale.** Today every tool lives inside this notebook. Next
   quarter another team wants `get_incident` too. The answer: tools as standardised,
   discoverable *servers* any agent can attach to. → **Week 8: MCP.**

## Exercise (take-home) · one per difficulty level

- **🟢 Easy:** give `status_bot` a second tool, `list_services()` (no arguments — note how the
  schema changes), and ask it *"what's broken right now?"* — it should enumerate services and
  check each. Bonus: return the answer as a `list[StatusReport]` via `output_type`.
- **🟡 Moderate:** add `resolve_incident(incident_id, resolution_note)` to the Claude loop:
  write the function (typed, docstring-first, `dict` return, note ≥ 15 chars, reject
  already-Resolved), hand-write its schema into `CLAUDE_TOOLS`, register it, and test both the
  happy path and a refusal (*"close INC-1042"* with no evidence — do your prompt rules push back?).
- **🔴 Difficult:** make `flag_change` require a `cr_approver` parameter, and extend
  `get_dependent_services` to walk **two** hops (dependents of dependents) — then rerun the
  session and watch whether the blast-radius reasoning changes. You've just built a
  change-approval gate and a deeper graph traversal, one week early on both counts.

## Wrap-up · what you built today

- **The mechanism:** agent = LLM + tools + loop. The model plans; deterministic code executes.
  You saw the loop hidden (EASY), wrote it by hand and watched every step — including a
  parallel fan-out (MODERATE), then let the SDK run it at five-tool scale with session memory
  (DIFFICULT).
- **The contract:** tools are typed functions with docstring-first descriptions returning
  structured dicts — a discipline that transferred unchanged across every agent today, and
  will transfer unchanged to ADK in its Skill Boost lab.
- **The trade:** frameworks exchange ceremony for sessions, generated schemas, run histories
  and deployment. Same loop underneath, always.
- **The interfaces:** agents can speak in typed objects (`output_type`), not just prose —
  which is what makes them composable components rather than chatbots.

**Next week:** the risk assessor gets teammates. Triage, analysis and execution split into
specialised agents with a coordinator, wired to a real ServiceNow instance — with the audit
trail and human-in-the-loop gates today's write-tools were begging for.

**Week 8:** the tools you inlined today become an MCP server — standardised, discoverable,
shared across every agent in the building.